# 05 · Geografía: municipios, suelo y overlay
**Responsables:** P3 (municipios) y P4 (suelo) · **Código:** `src/geo.py`, `src/silver_municipios.py`, `src/silver_suelo.py`, `src/gold_suelo.py`
**Salidas:** `silver/municipios.geoparquet`, `silver/capacidad_uso.geoparquet`, `gold/suelo_municipio.parquet`, `gold/suelo_resumen.parquet`

**Qué hace**
1. Reconstruye los polígonos desde el WKT de bronze con el CRS registrado (EPSG:4326).
2. Cuenta geometrías nulas, vacías e inválidas, las repara con `make_valid` y deja solo partes poligonales.
3. **P3:** valida 123 municipios únicos y mide cuánto cambia el área por la simplificación a 75 m frente a `area_km2` oficial.
4. **P4:** valida las clases (III–VIII, ZU, CA) y compara el área calculada con la que reporta el IGAC.
5. **Overlay (P4):** reproyecta ambas capas a **EPSG:9377** (metros), disuelve el suelo por clase (de unos 958 polígonos a unos 8) e intersecta con los municipios. Calcula el % de cada clase sobre el área cubierta y el % sin clasificar.

**Limitaciones que se declaran:** escala 1:100.000 y año 2004. Es capacidad de uso, no fertilidad actual. La clase predominante no implica que todo el municipio tenga ese suelo.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
from src.silver_municipios import construir_municipios
pob = pl.read_parquet(SILVER / "poblacion.parquet")
mun = construir_municipios(pob)
mun.head()

,codigo_municipio,municipio,area_km2,area_simplificada_km2,geometry
0,15001,Tunja,119.481492,119.236767,"POLYGON ((-73.41186 5.56682, -73.41006 5.56651..."
1,15022,Almeida,57.574192,57.494088,"POLYGON ((-73.37179 5.01386, -73.36445 5.01426..."
2,15047,Aquitania,940.354523,940.817515,"POLYGON ((-72.76346 5.64502, -72.76286 5.63893..."
3,15051,Arcabuco,137.669050,137.667092,"POLYGON ((-73.47156 5.8216, -73.47348 5.81683,..."
4,15087,Belén,162.774179,162.704152,"POLYGON ((-72.81735 6.01592, -72.81445 6.02122..."


In [3]:
from src.silver_suelo import construir_capacidad_uso
suelo = construir_capacidad_uso()
suelo["clase"].value_counts()

clase
VII     309
IV      287
VI      122
VIII    121
III      75
ZU       23
V        14
CA        7
Name: count, dtype: int64

In [4]:
from src.gold_suelo import composicion_suelo, distribucion_departamental
largo, resumen = composicion_suelo(mun, suelo)
resumen.sort("pct_sin_clasificar", descending=True).head(10)

codigo_municipio,area_municipio_ha,area_clasificada_ha,pct_sin_clasificar,clase_predominante,pct_clase_predominante,pct_clases_iii_iv
str,f64,f64,f64,str,f64,f64
"""15238""",23351.042185,21281.75694,8.86164,"""VIII""",50.881144,12.749808
"""15183""",67577.355597,61758.199134,8.611104,"""VIII""",80.596563,1.202297
"""15753""",12261.10151,11230.789329,8.403096,"""VIII""",54.680774,13.686871
"""15533""",44271.022636,40785.949241,7.872132,"""VIII""",53.408542,4.517142
"""15693""",11635.980931,10758.47492,7.541315,"""VIII""",50.819583,6.970479
"""15686""",6796.941621,6376.154332,6.190833,"""VII""",45.758383,25.732928
"""15162""",6366.623667,5991.087906,5.898507,"""VII""",45.875167,30.941715
"""15322""",3610.537784,3434.382711,4.878915,"""VI""",44.344281,0.0
"""15469""",21782.71905,20732.488115,4.821395,"""VI""",77.634869,13.169184


In [5]:
from src import graficos
graficos.distribucion_clases(distribucion_departamental(largo).to_pandas())

PosixPath('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/proyecto_boyaca/outputs/figures/p4_distribucion_clases.png')

In [6]:
pl.read_csv(QUALITY / "metricas_silver.csv").filter(pl.col("fuente").is_in(["municipios", "capacidad_uso"]))

fecha_ejecucion,capa,fuente,regla,descripcion,revisados,afectados,resultado,accion
str,str,str,str,str,i64,i64,str,str
"""2026-11-05T12:51:55""","""silver""","""municipios""","""codigo_divipola_valido""","""Código de 5 dígitos que inicia…",123,0,"""OK""","""rechazar"""
"""2026-11-05T12:51:55""","""silver""","""municipios""","""clave_unica""","""Un polígono por municipio""",123,0,"""OK""","""detener"""
"""2026-11-05T12:51:55""","""silver""","""municipios""","""cardinalidad""","""Se esperaban 123 municipios; h…",123,0,"""OK""","""alertar"""
"""2026-11-05T12:51:55""","""silver""","""municipios""","""geometrias_nulas_o_vacias""","""Sin geometría no hay polígono …",123,0,"""OK""","""rechazar"""
"""2026-11-05T12:51:55""","""silver""","""municipios""","""geometrias_invalidas_reparadas""","""Autointersecciones u otros err…",123,0,"""OK""","""alertar"""
…,…,…,…,…,…,…,…,…
"""2026-11-05T12:51:57""","""silver""","""capacidad_uso""","""geometrias_invalidas_reparadas""","""Autointersecciones u otros err…",958,2,"""FALLA""","""alertar"""
"""2026-11-05T12:51:58""","""silver""","""capacidad_uso""","""geometrias_irreparables""","""Tras reparar no quedó un políg…",958,0,"""OK""","""rechazar"""
"""2026-11-05T12:51:58""","""silver""","""capacidad_uso""","""tipo_geometria_poligonal""","""Tipos: {'Polygon': 958}""",958,0,"""OK""","""alertar"""
